# TEKNOFEST 2026 — Variant Pathogenicity Classifier
## Phases 0 → 3  (Setup · EDA-prep · Feature Engineering · Encoding)

This notebook is the **Colab driver** for the first four phases of the pipeline. All
heavy lifting lives in the importable `src/` package; cells here just orchestrate it
so the work is reproducible and modular.

**Scope (stops exactly after Phase 3):**
- **Phase 0** — checksums, load + merge the 4 CSVs, `variant_fingerprint`, **10% canary**
  hold-out (`StratifiedShuffleSplit` on label+source), **5-fold `StratifiedGroupKFold`**
  indices (group = fingerprint, stratify = label+source).
- **Phase 1 (data-prep half)** — feature-schema build + the CAT-column leakage audit that
  drives the encoding decisions. (The full visual EDA lives in `01_eda.ipynb` / `03_adversarial_validation.ipynb`.)
- **Phase 2** — stateless feature suite (missingness meta + tiers, AA physicochemical:
  Grantham/BLOSUM45-62-80/volume/polarity/…, population-frequency log-aggregations) and the
  leakage-free fit/transform parts (EK directionality-correction → rank-normalisation,
  missingness-pattern similarity).
- **Phase 3** — encoding: native-`NaN` tree matrix (no imputation), K-fold target encoding
  with **adaptive noise**, one-hot for genotypes, calibration utilities.

> **Colab notes:** all paths are relative to the repo root. No `!pip install` is issued —
> Colab already ships pandas/numpy/scikit-learn/pyarrow. GPU is auto-detected and wired into
> the Phase-4 model params (`config.model_gpu_params`); Phases 0–3 themselves are CPU data-prep.

In [1]:
# --- Environment setup -------------------------------------------------------
# Make every relative ./data and ./outputs path resolve by running from the repo
# root, and put the repo root on sys.path so `from src import ...` works whether
# the notebook is launched from ./ or ./notebooks.
import os
import sys


def find_repo_root(start: str = None) -> str:
    p = start or os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "data", "raw")) and os.path.isdir(os.path.join(p, "src")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return start or os.getcwd()


ROOT = find_repo_root()
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd

from src import config as C
from src import data_setup as ds
from src import feature_engineering as fe
from src import encoders as enc

C.set_global_seed(C.SEED)
C.ensure_dirs()

print("Repo root      :", os.getcwd())
print("Python         :", sys.version.split()[0])
print("pandas / numpy :", pd.__version__, "/", np.__version__)
print("GPU available  :", C.detect_gpu(), "(used by Phase-4 models, not Phase 0-3)")

Repo root      : C:\Users\fikre\Desktop\Teknofest2026
Python         : 3.12.6
pandas / numpy : 2.3.3 / 2.3.3
GPU available  : True (used by Phase-4 models, not Phase 0-3)


## Phase 0 — Environment, Data Integrity, Canary & Folds

`run_phase0()` performs, in order: seed → create output dirs → **SHA-256 checksums** of every
raw CSV (`outputs/checksums.txt`, re-run each session to detect drift) → load + tag + merge the
4 CSVs → add `variant_fingerprint` → carve the **10% canary** hold-out → generate **5 fold
indices**.

**Why `variant_fingerprint == Variant_ID`** (deviation from the plan's content-hash, justified by EDA):
the AA+EK content hash *splits* 567 rows that share a `Variant_ID` and *collides* ~370 sparse
"skeleton" rows into one giant group. `Variant_ID` is clean (3,224 unique, 0 within-ID label
conflicts), so it is the correct grouping key. The content hash is still available via
`ds.compute_content_fingerprint` for the Phase-9 ablation.

In [2]:
# --- Phase 0: run the full setup pipeline ------------------------------------
artifacts = ds.run_phase0(enforce_group_disjoint=True)

development = artifacts["development"]   # all modelling happens on THIS
canary = artifacts["canary"]             # never touched until final evaluation
folds = artifacts["folds"]               # list of {fold, train[], val[]}

# Sanity: the 5 validation folds must exactly partition the development set,
# and the canary must share zero variant fingerprints with development.
val_union = sorted(sum([f["val"] for f in folds], []))
assert val_union == list(range(len(development))), "folds must partition development"
assert not (set(canary[C.FINGERPRINT_COL]) & set(development[C.FINGERPRINT_COL])), "canary leak!"
print("\nArtifacts written:")
for p in [C.CHECKSUMS_PATH, C.MERGED_PATH, C.DEVELOPMENT_PATH, C.CANARY_PATH, C.FOLD_INDICES_PATH]:
    print(f"  {'OK ' if os.path.exists(p) else 'MISS'} {p}")

[checksum] YARISMA_TRAIN_MASTER.csv: c9d9bf6fc6f5abe123ca90cc52fa83b2a4f63b69d929bff17d73c4ab9c0348a7
[checksum] YARISMA_TRAIN_KANSER.csv: bc695838a7e6b77413bfa7c0725a3907b15edc6a2f0cb0da3ae9bdc8f5a905cb
[checksum] YARISMA_TRAIN_PAH.csv: 554e4fd68f8108a6b6ef11ef7fe4ccce932b859464b6eb9b5474e7681849d8c9
[checksum] YARISMA_TRAIN_CFTR.csv: c05205e31cc0d1a9edac4558f8a8ed78aea1f2a5a248f48bd71b0cd22af467d0


[load] master   YARISMA_TRAIN_MASTER.csv   shape=(2931, 354)
[load] kanser   YARISMA_TRAIN_KANSER.csv   shape=(388, 354)
[load] pah      YARISMA_TRAIN_PAH.csv      shape=(372, 354)
[load] cftr     YARISMA_TRAIN_CFTR.csv     shape=(111, 354)
[load] merged shape=(3802, 354) | pos_rate=0.7409
[fingerprint] unique=3224 | variants spanning >1 source=578 | rows flagged duplicated=1156


[canary] moved 90 group-overlapping rows back to development


[canary] canary n=291 pos_rate=0.6942 -> ./data/processed\canary_set.parquet
[canary] development n=3511 pos_rate=0.7448 -> ./data/processed\development_set.parquet
[canary] fingerprint overlap canary<->development = 0 (target 0)


[folds] fold 0: train=2812 val=699 val_pos_rate=0.758 groups(tr/va)=2356/588 leak=0
[folds] fold 1: train=2815 val=696 val_pos_rate=0.728 groups(tr/va)=2355/589 leak=0
[folds] fold 2: train=2804 val=707 val_pos_rate=0.727 groups(tr/va)=2355/589 leak=0
[folds] fold 3: train=2810 val=701 val_pos_rate=0.748 groups(tr/va)=2355/589 leak=0
[folds] fold 4: train=2803 val=708 val_pos_rate=0.763 groups(tr/va)=2355/589 leak=0
[folds] saved -> ./outputs/folds\fold_indices.json

Artifacts written:
  OK  ./outputs\checksums.txt
  OK  ./data/processed\merged_full.parquet
  OK  ./data/processed\development_set.parquet
  OK  ./data/processed\canary_set.parquet
  OK  ./outputs/folds\fold_indices.json


## Phase 1 (data-prep half) — Feature Schema & CAT Leakage Audit

Two things from Phase 1 are *required* before Phases 2–3 and are reproduced here:

1. **Feature schema** — classify the 351 raw features into AL/EK/CAT/AA, and split AL into
   **binary flags (77)** vs **continuous scores (257)**. This is a *structural* property (no
   labels), so it is safe to derive from the full development set; we persist it so val/test
   reuse identical column lists.
2. **CAT-column leakage audit** (Top-1 priority in the plan) — crosstab each CAT column vs the
   label. Findings: **CAT_1 = gnomAD population**, **CAT_2 = All-of-Us population** (ancestry
   membership ⇒ *safe* to target-encode); **CAT_3/4/5 = genotype calls** ⇒ one-hot; **CAT_6 =
   genomic-region mask** (segdup/lcr, 98% missing) ⇒ one-hot. None is a pathogenicity proxy.

In [3]:
# --- Phase 1: build + persist the feature schema -----------------------------
schema = fe.build_feature_schema(development)
fe.save_feature_schema(schema)
print(f"schema -> AL={len(schema.al_cols)} (binary={len(schema.binary_al_cols)}, "
      f"continuous={len(schema.continuous_al_cols)}) | EK={len(schema.ek_cols)} | "
      f"CAT={len(schema.cat_cols)} | AA={len(schema.aa_cols)} | AF block={len(schema.af_cols)}")
assert len(schema.feature_cols) == C.N_RAW_FEATURES  # 351

# --- Phase 1.3: CAT-column leakage audit (decisions live in config.CAT_ENCODING_STRATEGY)
print("\nCAT leakage audit (share Pathogenic per category; flag >95% pure):")
for col in schema.cat_cols:
    s = development[col]
    ct = pd.crosstab(s, development[C.LABEL_COL], normalize="index")
    pure = int(((ct.get(1, 0) >= 0.95) | (ct.get(1, 0) <= 0.05)).sum()) if 1 in ct.columns else 0
    print(f"  {col:6s} n_unique={s.nunique():3d}  missing={s.isna().mean()*100:4.0f}%  "
          f">95%-pure cats={pure:2d}  -> strategy='{C.CAT_ENCODING_STRATEGY[col]}'")

schema -> AL=334 (binary=77, continuous=257) | EK=9 | CAT=6 | AA=2 | AF block=26

CAT leakage audit (share Pathogenic per category; flag >95% pure):
  CAT_1  n_unique= 30  missing=  37%  >95%-pure cats= 6  -> strategy='target'
  CAT_2  n_unique=  7  missing=  60%  >95%-pure cats= 0  -> strategy='target'
  CAT_3  n_unique=  5  missing=  10%  >95%-pure cats= 0  -> strategy='onehot'
  CAT_4  n_unique=  5  missing=  10%  >95%-pure cats= 0  -> strategy='onehot'
  CAT_5  n_unique=  5  missing=  10%  >95%-pure cats= 0  -> strategy='onehot'
  CAT_6  n_unique=  3  missing=  98%  >95%-pure cats= 1  -> strategy='onehot'


## Phase 2 — Feature Engineering

**Stateless / row-level** features (no labels, no training stats — computed once on the whole
development set and reused across folds):
- **2.1 Missingness meta** — `n_missing_total / _al / _ek`, `pct_missing`, 3-way
  `missingness_tier`, presence flags, **block-missingness** (`block_*_present`, `n_blocks_present`,
  `tool_coverage_profile`).
- **2.7 Tier flags** — `is_skeleton_row` (≥337 missing), `is_dense_row` (≤20 missing).
- **2.2 Amino-acid suite** — BLOSUM45/62/80 (+mean/min), Grantham (+category), the full
  physicochemical set for AA_1/AA_2 (MW, 2× hydrophobicity, pI, charge, volume, polarity, helix/
  sheet/coil, SASA, flexibility) with signed/abs **deltas** + `max_abs_delta`, chemical-group
  membership, `group_change`, `charge_direction_change`, and special flags (`is_stop_gain`,
  `is_stop_loss`, `is_synonymous`, non-standard, `transition_compatible`, `aa_pair`).
- **2.3 Population frequency** — log10 aggregations (`af_log_max/mean/std/min`, spread, rarity flags).
- **2.4 In-silico consensus** — damaging/benign vote counts & ratios over the 77 binary flags.
- **2.8 Source flags** — `is_master/kanser/pah/cftr`.

In [4]:
# --- Phase 2: stateless features (computed once on the full development set) --
stateless_dev = fe.engineer_features_stateless(development, schema)
new_cols = [c for c in stateless_dev.columns if c not in development.columns]
print(f"stateless features added: {len(new_cols)}  (development now {stateless_dev.shape[1]} cols)")

# A quick peek at representative engineered features grouped by family.
peek = ["n_missing_total", "missingness_tier", "is_skeleton_row", "is_dense_row",
        "blosum62_score", "blosum_min", "grantham_score", "grantham_category",
        "delta_volume", "delta_charge", "charge_direction_change", "max_abs_delta",
        "is_stop_gain", "is_synonymous", "transition_compatible", "aa_pair",
        "af_log_max", "is_rare", "damaging_vote_ratio", "n_tools_available",
        "tool_coverage_profile", "is_pah"]
stateless_dev[peek].head(4)

stateless features added: 124  (development now 480 cols)


,n_missing_total,missingness_tier,is_skeleton_row,is_dense_row,blosum62_score,blosum_min,grantham_score,grantham_category,delta_volume,delta_charge,...,is_stop_gain,is_synonymous,transition_compatible,aa_pair,af_log_max,is_rare,damaging_vote_ratio,n_tools_available,tool_coverage_profile,is_pah
0,217,partial,0,0,1.0,1.0,31.780001,conservative,19.0,0.0,...,0,0,0.0,V_L,-1.964109,0.0,1.000000,35,11111101,0
1,2,dense,0,1,1.0,1.0,56.869999,moderate,26.0,2.0,...,0,0,1.0,E_K,-1.029536,0.0,0.987013,77,11111101,0
2,351,sparse,1,0,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,?_?,NaN,NaN,NaN,0,00000000,0
3,245,partial,0,0,2.0,1.0,44.599998,conservative,18.0,0.0,...,0,0,0.0,D_E,-2.204324,0.0,1.000000,15,11111101,0


### Phase 2 (cont.) — Leakage-free fit/transform + per-fold dataset builder

Everything that needs **labels or training-set statistics** is fit on the *training rows of the
current fold only*:
- **2.5 EK scores** — `EKRankNormalizer` auto-detects each EK column's **directionality** from the
  training-fold point-biserial correlation (inverts SIFT-like columns), then **rank-normalises**
  to a training-set percentile. Aggregates (`ek_mean/max/min/range/std_ranked`, `ek_unanimity`,
  `ek_strong_damaging`, `ek_conflicting`) follow.
- **2.1 (extra)** — `MissingnessPatternEncoder`: cosine similarity of each row's missingness
  vector to the per-class missingness centroid learned on train.
- **2.6 Interactions** — cross-terms (score × AA severity × frequency × completeness) built after
  the EK aggregates exist.

The `build_fold_dataset` helper chains stateless → fit-on-train → transform → Phase-3 encoding,
returning the leakage-free `(X_train, y_train, X_val, y_val)` that the Phase-4 CV loop will consume.

In [5]:
# --- Phase 2+3: leakage-free per-fold dataset builder ------------------------
from collections import namedtuple

FoldData = namedtuple(
    "FoldData",
    ["X_train", "y_train", "X_val", "y_val", "feature_cols", "fitted", "encoders", "train_enc", "val_enc"],
)


def build_fold_dataset(fold_id: int, stateless_df: pd.DataFrame = None, smoothing: float = 10.0) -> FoldData:
    """Chain Phase-2 fit/transform + Phase-3 encoding for one fold, fitting every
    label/stat-dependent transform on the TRAIN rows only (no leakage)."""
    sl = stateless_dev if stateless_df is None else stateless_df
    f = folds[fold_id]
    tr = sl.iloc[f["train"]].reset_index(drop=True)
    va = sl.iloc[f["val"]].reset_index(drop=True)
    y_tr, y_va = tr[C.LABEL_COL], va[C.LABEL_COL]

    # Phase 2 fit-on-train -> transform both splits
    fitted = fe.fit_fold(tr, y_tr, schema)
    tr_f = fe.transform_split(tr, fitted)
    va_f = fe.transform_split(va, fitted)

    # Phase 3 encode-on-train -> transform both splits
    tr_e, va_e, encoders = enc.encode_fold(tr_f, va_f, y_tr, fold_id=fold_id, smoothing=smoothing)

    # Native-NaN tree matrix (no imputation)
    feat_cols = enc.get_tree_feature_columns(tr_e)
    X_tr = enc.assemble_tree_matrix(tr_e, feat_cols)
    X_va = enc.assemble_tree_matrix(va_e, feat_cols)
    return FoldData(X_tr, y_tr, X_va, y_va, feat_cols, fitted, encoders, tr_e, va_e)


# Demonstrate on fold 0
fold0 = build_fold_dataset(0)
print("EK directionality (train-fold corr; -1 = SIFT-like inverted):")
print(" ", fold0.fitted.ek_normalizer.directions)
ek1 = fold0.X_train["EK_1_ranked"].dropna()
print(f"EK_1_ranked range = [{ek1.min():.3f}, {ek1.max():.3f}]  (percentile in [0,1])")
print(f"fold0 tree matrix -> X_train {fold0.X_train.shape}, X_val {fold0.X_val.shape}, "
      f"#features={len(fold0.feature_cols)}")

EK directionality (train-fold corr; -1 = SIFT-like inverted):
  {'EK_1': 1, 'EK_2': 1, 'EK_3': 1, 'EK_4': 1, 'EK_5': 1, 'EK_6': 1, 'EK_7': 1, 'EK_8': 1, 'EK_9': 1}
EK_1_ranked range = [0.000, 1.000]  (percentile in [0,1])
fold0 tree matrix -> X_train (2812, 522), X_val (699, 522), #features=522


## Phase 3 — Encoding & Missing-Value Strategy

- **3.1-A (primary)** — **native `NaN`**: the tree matrix preserves missing values so
  XGBoost (`tree_method='hist'`) and LightGBM learn the optimal default split direction. **No
  imputation.**
- **3.3** — **K-fold target encoding with adaptive noise** (`σ = 0.01/√count`) for the *safe*
  population columns (CAT_1, CAT_2), the high-cardinality `aa_pair`, and `source`. Inner K-fold
  produces OOF encodings for train; the full-train smoothed mapping (no noise) encodes val/test;
  mappings are persisted per fold.
- **3.2** — **one-hot** for the small/safe genotype + region columns (CAT_3/4/5/6); stateless AA
  label encoding.
- **3.1-B / 3.4** — sentinel & fold-median imputation (for the Phase-9.4 ablation) and Isotonic/
  Platt calibration helpers (used later, in Phase 6) are available in `src/encoders.py`.

The cell below inspects fold 0's encoded outputs and confirms the leakage-free guarantees.

In [6]:
# --- Phase 3: inspect fold-0 encodings & verify the guarantees ---------------
te_cols = [c for c in fold0.train_enc.columns if c.endswith("_te")]
oh_cols = [c for c in fold0.train_enc.columns if "_oh_" in c]
print("target-encoded :", te_cols)
print(f"one-hot columns: {len(oh_cols)} (genotype CAT_3/4/5 + region CAT_6)")

# (1) native NaN preserved -> no imputation in the tree matrix
nan_frac = float(fold0.X_train.isna().mean().mean())
assert fold0.X_train.isna().any().any(), "expected NaN to be preserved"
# (2) train/val share identical feature columns
assert list(fold0.X_train.columns) == list(fold0.X_val.columns)
# (3) no bookkeeping / raw-string / label columns leaked into features
for bad in [C.LABEL_COL, C.ID_COL, C.SOURCE_COL, "AA_1", "AA_2", "aa_pair", "missingness_tier"]:
    assert bad not in fold0.feature_cols
print(f"\nOK | mean NaN fraction in tree matrix = {nan_frac:.2%} (preserved, not imputed)")
print("OK | train/val feature columns identical; no label/ID/string leakage")
print(f"OK | per-fold TE mapping persisted -> {C.TE_MAPPINGS_DIR}/fold_0_CAT_1.json")

# Secondary missing-value strategies (Phase 3.1-B, for the Phase 9.4 ablation)
X_sentinel = enc.sentinel_impute(fold0.X_train)                       # -999 marker
X_median = enc.FoldMedianImputer().fit(fold0.X_train).transform(fold0.X_train)  # +indicators
print(f"\nablation-ready: sentinel(no-NaN={not X_sentinel.isna().any().any()}) | "
      f"median+indicators(cols={X_median.shape[1]})")

target-encoded : ['CAT_1_te', 'CAT_2_te', 'aa_pair_te', 'source_te']
one-hot columns: 22 (genotype CAT_3/4/5 + region CAT_6)

OK | mean NaN fraction in tree matrix = 38.87% (preserved, not imputed)
OK | train/val feature columns identical; no label/ID/string leakage
OK | per-fold TE mapping persisted -> ./outputs/encoders/target_encoding_mappings/fold_0_CAT_1.json



ablation-ready: sentinel(no-NaN=True) | median+indicators(cols=1044)


### Phase 3 (cont.) — Fit the *final* (test-time) encoders on the full development set

For CV, encoders are refit per fold (above). For the eventual **test-time** pipeline we fit them
**once on the entire development set** and persist them, so the hidden test set (and the canary,
touched only at the very end) are transformed with exactly the same statistics. This satisfies
the reproducibility checklist (EK direction map + rank thresholds + target-encoding mappings on
disk). We transform the canary here **only to materialise its feature matrix** — its labels are
not used for any fitting.

In [7]:
# --- Phase 3: fit final encoders on FULL development, persist, transform canary
y_dev = development[C.LABEL_COL]

# Phase 2 fit on all development rows
final_fit = fe.fit_fold(stateless_dev, y_dev, schema)
final_fit.ek_normalizer.save()  # -> ek_direction_map.json + ek_rank_thresholds.json
dev_feat = fe.transform_split(stateless_dev, final_fit)

# Canary: stateless -> transform with the development-fitted objects (no fitting on canary)
canary_sl = fe.engineer_features_stateless(canary, schema)
canary_feat = fe.transform_split(canary_sl, final_fit)

# Phase 3 encoders fit on all development; fold_id='full' tags the saved mappings
dev_enc, canary_enc, final_encoders = enc.encode_fold(
    dev_feat, canary_feat, y_dev, fold_id="full")

final_feature_cols = enc.get_tree_feature_columns(dev_enc)
X_dev = enc.assemble_tree_matrix(dev_enc, final_feature_cols)
X_canary = enc.assemble_tree_matrix(canary_enc, final_feature_cols)

# Persist the materialised matrices + the canonical feature list for Phase 4+
import json
X_dev.assign(**{C.LABEL_COL: y_dev.values, C.SOURCE_COL: development[C.SOURCE_COL].values}) \
     .to_parquet(os.path.join(C.PROCESSED_DIR, "development_features.parquet"), index=False)
X_canary.assign(**{C.LABEL_COL: canary[C.LABEL_COL].values, C.SOURCE_COL: canary[C.SOURCE_COL].values}) \
        .to_parquet(os.path.join(C.PROCESSED_DIR, "canary_features.parquet"), index=False)
with open(os.path.join(C.ENCODERS_DIR, "final_feature_list.json"), "w") as fh:
    json.dump(final_feature_cols, fh, indent=2)

print(f"final feature matrix: development {X_dev.shape} | canary {X_canary.shape}")
print(f"#features = {len(final_feature_cols)}")
print("EK direction map:", final_fit.ek_normalizer.directions)

final feature matrix: development (3511, 522) | canary (291, 522)
#features = 522
EK direction map: {'EK_1': 1, 'EK_2': 1, 'EK_3': 1, 'EK_4': 1, 'EK_5': 1, 'EK_6': 1, 'EK_7': 1, 'EK_8': 1, 'EK_9': 1}


In [8]:
# --- Artifacts summary (end of Phase 3) --------------------------------------
print("Phase 0-3 complete. Persisted artifacts:\n")
expected = [
    C.CHECKSUMS_PATH,
    C.MERGED_PATH, C.DEVELOPMENT_PATH, C.CANARY_PATH,
    C.FOLD_INDICES_PATH,
    C.FEATURE_SCHEMA_PATH, C.EK_DIRECTION_PATH, C.EK_RANK_THRESHOLDS_PATH,
    os.path.join(C.ENCODERS_DIR, "final_feature_list.json"),
    os.path.join(C.PROCESSED_DIR, "development_features.parquet"),
    os.path.join(C.PROCESSED_DIR, "canary_features.parquet"),
    os.path.join(C.TE_MAPPINGS_DIR, "fold_0_CAT_1.json"),
    os.path.join(C.TE_MAPPINGS_DIR, "fold_full_CAT_1.json"),
]
for p in expected:
    flag = "OK " if os.path.exists(p) else "MISS"
    print(f"  [{flag}] {p}")
print("\nNext: Phase 4 (default models) loads outputs/folds/fold_indices.json and rebuilds each "
      "fold via build_fold_dataset(fold_id), then trains GPU XGB/LGBM/CatBoost. Stopping here.")

Phase 0-3 complete. Persisted artifacts:

  [OK ] ./outputs\checksums.txt
  [OK ] ./data/processed\merged_full.parquet
  [OK ] ./data/processed\development_set.parquet
  [OK ] ./data/processed\canary_set.parquet
  [OK ] ./outputs/folds\fold_indices.json
  [OK ] ./outputs/encoders\feature_schema.json
  [OK ] ./outputs/encoders\ek_direction_map.json
  [OK ] ./outputs/encoders\ek_rank_thresholds.json
  [OK ] ./outputs/encoders\final_feature_list.json
  [OK ] ./data/processed\development_features.parquet
  [OK ] ./data/processed\canary_features.parquet
  [OK ] ./outputs/encoders/target_encoding_mappings\fold_0_CAT_1.json
  [OK ] ./outputs/encoders/target_encoding_mappings\fold_full_CAT_1.json

Next: Phase 4 (default models) loads outputs/folds/fold_indices.json and rebuilds each fold via build_fold_dataset(fold_id), then trains GPU XGB/LGBM/CatBoost. Stopping here.
